# EKC Analýza — Příprava dat pro Power BI
## Datový pipeline: od surových CSV souborů po vizualizaci v Power BI

---

### Co tento notebook dělá

Tento notebook **neprodukuje grafy** — ty vytvoříme v Power BI.  
Jeho úkolem je:
1. **Načíst a propojit** zdrojová data
2. **Vypočítat statistiky** (korelace, regrese), které Power BI nativně neumí
3. **Exportovat čisté CSV soubory** optimalizované pro Power BI datový model

### Výzkumné otázky

| Otázka | Zaměření | Výstupní soubor |
|---|---|---|
| **Q1** | Platí EKC globálně? Vztah HDP vs. změna lesa | `Q1_ekc_crosssection.csv` |
| **Q2** | Záleží na regionu více než na HDP? | `Q2_regional_panel.csv` |
| **Q3** | Kde jsou největší výjimky? Role policy faktorů | `Q3_outliers_policy.csv` |
| **Statistiky** | Korelace a regrese pro Power BI cards | `stats_summary.csv` |

### Vstupní soubory

| Soubor | Obsah |
|---|---|
| `data_dan/forest_ekc_model.csv` | Lesnatost 1990 a 2025, změna v p.b. |
| `data_dan/MAIN_Forest_GDP_joined.csv` | Panel data: lesnatost + HDP po letech |
| `data_raw/2025_World_Bank_classification_by_Income.csv` | Příjmová skupina + region |
| `data_raw/Forest_Policy_Legislation.csv` | Existence lesních politik a legislativ |

## 1. Nastavení — Import knihoven

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
from scipy import stats
import os

# Cesty k datovým složkám
DATA_DAN = '../data_dan/'
DATA_RAW = '../data_raw/'
OUTPUT   = './'  # výstupní CSV soubory uložíme vedle tohoto notebooku

# exist_ok=True zajistí, že funkce nevyhodí chybu, pokud složka už existuje
os.makedirs(OUTPUT, exist_ok=True)
print('Setup OK.')

Setup OK.


## 2. Načtení surových dat

Načteme všechna zdrojová data a ihned zkontrolujeme jejich strukturu.  
Cílem je zjistit: **jaké sloupce máme, kolik řádků, co chybí**.

In [2]:
# --- Zdroj 1: Změna lesnatosti 1990-2025 ---
# Jeden řádek = jedna země
# Sloupce: Country, Code, Forest_1990, Forest_2025, Forest_change
df_forest = pd.read_csv(DATA_DAN + 'forest_ekc_model.csv')
print(f'[forest]  {df_forest.shape[0]} zemí, {df_forest.shape[1]} sloupců')
df_forest.head(3)

[forest]  211 zemí, 5 sloupců


,Country,Code,Forest_1990,Forest_2025,Forest_change
0,Afghanistan,AFG,1.85,1.85,0.00
1,Albania,ALB,28.79,34.34,5.55
2,Algeria,DZA,0.72,0.71,-0.01


In [3]:
# --- Zdroj 2: Panel data — lesnatost + HDP po letech ---
# Jeden řádek = jedna země v jednom roce (long format)
df_panel = pd.read_csv(DATA_DAN + 'MAIN_Forest_GDP_joined.csv')

# Přejmenujeme dlouhé názvy sloupců pro pohodlnější práci
df_panel = df_panel.rename(columns={
    'Entity': 'Country',
    'Share of land covered by forest': 'Forest_share',
    'GDP per capita (current US$)': 'GDP_per_capita'
})

# Sloupec s anotacemi nepotřebujeme — odstraníme ho
# errors='ignore' = nevyhodí chybu, pokud sloupec neexistuje
df_panel = df_panel.drop(columns=['Share of land covered by forest (Annotations)'], errors='ignore')

print(f'[panel]   {df_panel["Country"].nunique()} zemí × {df_panel["Year"].nunique()} let = {df_panel.shape[0]} řádků')
print(f'          Roky: {df_panel["Year"].min()} – {df_panel["Year"].max()}')
print(f'          Chybějící GDP: {df_panel["GDP_per_capita"].isnull().sum()} řádků')
df_panel.head(3)

[panel]   221 zemí × 120 let = 7970 řádků
          Roky: 1000 – 2025
          Chybějící GDP: 929 řádků


,Country,Code,Year,Forest_share,GDP_per_capita
0,Afghanistan,AFG,1990,1.854315,NaN
1,Afghanistan,AFG,1991,1.854315,NaN
2,Afghanistan,AFG,1992,1.854315,NaN


In [4]:
# --- Zdroj 3: Klasifikace zemí dle příjmu (World Bank) ---
# skiprows=2 přeskočí textové záhlaví souboru (nejsou to datové řádky)
df_income = pd.read_csv(
    DATA_RAW + '2025_World_Bank_classification_by_Income.csv',
    skiprows=2
)

# Ponecháme jen sloupce, které opravdu potřebujeme
df_income = df_income[['Code', 'Region', 'Income group']].dropna(subset=['Code'])

# Přejmenujeme pro konzistenci s ostatními datasety
df_income = df_income.rename(columns={'Income group': 'Income_group'})

print(f'[income]  {len(df_income)} zemí')
print('Příjmové skupiny:', df_income['Income_group'].value_counts().to_dict())
df_income.head(3)

[income]  218 zemí
Příjmové skupiny: {'High income': 86, 'Upper middle income': 55, 'Lower middle income': 51, 'Low income': 26}


,Code,Region,Income_group
0,AFG,South Asia,Low income
1,ALB,Europe & Central Asia,Upper middle income
2,DZA,Middle East & North Africa,Upper middle income


In [5]:
# --- Zdroj 4: Forest Policy & Legislation ---
# Sloupce: regions, iso3, name, National/Sub-national policies, legislations, platform, traceability
# encoding='utf-8-sig' ošetří BOM znak na začátku některých UTF-8 souborů
df_policy = pd.read_csv(
    DATA_RAW + 'Forest_Policy_Legislation.csv',
    encoding='utf-8-sig'
)

# Zkrátíme názvy sloupců
df_policy = df_policy.rename(columns={
    'iso3':  'Code',
    'name':  'Country_policy',
    'regions': 'Region_policy',
    'National policies supporting SFM':               'Policy_national',
    'National legislations supporting SFM':           'Legislation_national',
    'National platform for stakeholder participation': 'Platform_national',
    'National existence of traceability system':      'Traceability_national'
})

print(f'[policy]  {len(df_policy)} zemí')
print('Sloupce:', df_policy.columns.tolist())

[policy]  236 zemí
Sloupce: ['Region_policy', 'subregions', 'Code', 'iso2', 'm49', 'Country_policy', 'Desk study', 'Policy_national', 'Sub-national policies supporting SFM', 'Legislation_national', 'Sub-national legislations supporting SFM', 'Platform_national', 'Sub-national platform for stakeholder participation', 'Traceability_national', 'Sub-national existence of traceability system', 'comments']


## 3. Příprava dat pro Q1 — EKC cross-sectional dataset

**Výzkumná otázka Q1:**  
*Platí EKC teorie globálně? Existuje statisticky měřitelný vztah mezi HDP na obyvatele a změnou plochy lesa?*

**Struktura výstupního souboru:**  
Jeden řádek = jedna země. Každá země má:
- lesnatost v 1990 a 2025 + změnu
- průměrné HDP na obyvatele (průměr přes všechna dostupná léta)
- log transformaci HDP (pro EKC analýzu)
- příjmovou skupinu a region
- pořadové číslo příjmové skupiny (pro správné řazení v Power BI)
- příznak typu odlehlé hodnoty

In [6]:
# Krok 1: Průměrné HDP na obyvatele za všechna dostupná léta
#
# Nejprve odstraníme řádky bez HDP (NaN), aby průměr nebyl zkreslený.
# .groupby('Code') = seskupíme podle kódu země
# .mean()          = průměr přes všechna léta v každé skupině
# .reset_index()   = Code se stane normálním sloupcem (ne indexem)
df_gdp_avg = (
    df_panel
    .dropna(subset=['GDP_per_capita'])
    .groupby('Code')['GDP_per_capita']
    .mean()
    .reset_index()
    .rename(columns={'GDP_per_capita': 'GDP_avg'})
)

print(f'Počet zemí s průměrným HDP: {len(df_gdp_avg)}')

Počet zemí s průměrným HDP: 208


In [7]:
# Krok 2: Merge všech tří zdrojů dohromady
#
# how='inner' = zachová jen země přítomné v OBOU tabulkách
# how='left'  = zachová všechny řádky levé tabulky, chybějící doplní NaN
#
# Pořadí merge: df_forest (211 zemí) -> přidáme GDP -> přidáme income/region
q1 = (
    df_forest
    .merge(df_gdp_avg, on='Code', how='inner')   # odpadnou země bez GDP dat
    .merge(df_income,  on='Code', how='left')    # přidáme region + income group
)

# Odstraníme řádky kde chybí klíčové proměnné
q1 = q1.dropna(subset=['Forest_change', 'GDP_avg', 'Income_group'])

print(f'Po merge: {len(q1)} zemí')

Po merge: 199 zemí


In [8]:
# Krok 3: Odvozené sloupce

# Log transformace HDP
# Rozsah HDP je cca $300 (nejchudší) až $130 000 (nejbohatší) — log srovná škálu
q1['log_GDP'] = np.log(q1['GDP_avg'])

# Numerické pořadí příjmové skupiny
# Power BI potřebuje číslo pro správné řazení — bez toho řadí abecedně
income_order_map = {
    'Low income': 1,
    'Lower middle income': 2,
    'Upper middle income': 3,
    'High income': 4
}
# .map(slovník) nahradí každou hodnotu odpovídající hodnotou ze slovníku
q1['Income_group_order'] = q1['Income_group'].map(income_order_map)

# Příznak outlierů pomocí np.select()
# np.select() funguje jako série if/elif/else:
#   conditions = seznam podmínek (testovány v pořadí)
#   choices    = hodnoty pro každou splněnou podmínku
#   default    = hodnota pokud žádná podmínka neplatí
gdp_q25 = q1['GDP_avg'].quantile(0.25)  # 25. percentil = dolní čtvrtina zemí
gdp_q75 = q1['GDP_avg'].quantile(0.75)  # 75. percentil = horní čtvrtina zemí

conditions = [
    (q1['GDP_avg'] > gdp_q75) & (q1['Forest_change'] < -2),  # bohatý + odlesňuje
    (q1['GDP_avg'] < gdp_q25) & (q1['Forest_change'] >  2),  # chudý  + zalesňuje
]
choices = ['Rich & Deforesting', 'Poor & Reforesting']
q1['Outlier_type'] = np.select(conditions, choices, default='Expected pattern')

print('Rozložení outlier typů:')
print(q1['Outlier_type'].value_counts())

Rozložení outlier typů:
Outlier_type
Expected pattern      192
Poor & Reforesting      5
Rich & Deforesting      2
Name: count, dtype: int64


In [9]:
# Krok 4: Seřazení a export

# Seřadíme dle Income_group_order — zachová logické pořadí při importu do Power BI
q1 = q1.sort_values(['Income_group_order', 'Country']).reset_index(drop=True)

# index=False   = do CSV se neuloží automatický číselný index pandas
# encoding='utf-8-sig' = UTF-8 s BOM znakem — Excel a Power BI ho správně rozpoznají
q1.to_csv(OUTPUT + 'Q1_ekc_crosssection.csv', index=False, encoding='utf-8-sig')

print(f'Uloženo: Q1_ekc_crosssection.csv ({len(q1)} řádků)')
print('Sloupce:', q1.columns.tolist())
q1.head()

Uloženo: Q1_ekc_crosssection.csv (199 řádků)
Sloupce: ['Country', 'Code', 'Forest_1990', 'Forest_2025', 'Forest_change', 'GDP_avg', 'Region', 'Income_group', 'log_GDP', 'Income_group_order', 'Outlier_type']


,Country,Code,Forest_1990,Forest_2025,Forest_change,GDP_avg,Region,Income_group,log_GDP,Income_group_order,Outlier_type
0,Afghanistan,AFG,1.85,1.85,0.00,415.494946,South Asia,Low income,6.029470,1,Expected pattern
1,Burkina Faso,BFA,30.33,11.83,-18.50,529.079969,Sub-Saharan Africa,Low income,6.271140,1,Expected pattern
2,Burundi,BDI,10.77,10.89,0.12,194.791896,Sub-Saharan Africa,Low income,5.271932,1,Expected pattern
3,Central African Republic,CAF,74.54,72.39,-2.15,388.669107,Sub-Saharan Africa,Low income,5.962728,1,Expected pattern
4,Chad,TCD,5.34,2.87,-2.47,690.756362,Sub-Saharan Africa,Low income,6.537787,1,Expected pattern


## 4. Výpočet statistik pro Q1

Power BI nativně **neumí počítat Spearmanovu korelaci ani R² polynomiální regrese**.  
Spočítáme je v Pythonu a exportujeme jako tabulku — v Power BI ji zobrazíme jako **Card visual** nebo **Table**.

Výstup: `stats_summary.csv` — tabulka klíč–hodnota pro snadné použití v Power BI.

In [10]:
x = q1['log_GDP'].values
y = q1['Forest_change'].values

# --- Lineární regrese ---
# stats.linregress() vrátí 5 hodnot:
#   slope     = směrnice přímky
#   intercept = průsečík s osou Y
#   r_lin     = Pearsonův korelační koeficient
#   p_lin     = p-hodnota (je vztah statisticky signifikantní?)
#   _         = standard error (nepotřebujeme, použijeme _ jako zástupný symbol)
slope_lin, intercept_lin, r_lin, p_lin, _ = stats.linregress(x, y)
r2_lin = r_lin ** 2

# --- Kvadratická regrese ---
# np.polyfit(x, y, deg=2) fituje polynom stupně 2 na data
# Vrátí koeficienty od nejvyššího stupně: [c2, c1, c0]
coeffs  = np.polyfit(x, y, deg=2)
poly_fn = np.poly1d(coeffs)   # vytvoří funkci z koeficientů
y_pred  = poly_fn(x)          # predikované hodnoty

# Manuální výpočet R² pro polynomiální regresi:
# R2 = 1 - (suma čtverců reziduálů) / (celková variabilita)
ss_res  = np.sum((y - y_pred) ** 2)
ss_tot  = np.sum((y - y.mean()) ** 2)
r2_quad = 1 - ss_res / ss_tot

# --- Korelace ---
# Spearman je robustnější vůči outlierům než Pearson
# Obě funkce vrátí dvojici (koeficient, p-hodnota)
spearman_r, spearman_p = stats.spearmanr(x, y)
pearson_r,  pearson_p  = stats.pearsonr(x, y)

# --- Sestavení a export tabulky statistik ---
# Formát klic-hodnota-poznamka je nejflexibilnější pro Power BI Cards
stats_rows = [
    {'Stat': 'N countries',         'Value': len(q1),              'Note': 'countries in Q1 dataset'},
    {'Stat': 'Spearman r',          'Value': round(spearman_r, 4), 'Note': 'log_GDP vs Forest_change'},
    {'Stat': 'Spearman p-value',    'Value': round(spearman_p, 4), 'Note': '< 0.05 = significant'},
    {'Stat': 'Pearson r',           'Value': round(pearson_r, 4),  'Note': 'log_GDP vs Forest_change'},
    {'Stat': 'Linear R2',           'Value': round(r2_lin, 4),     'Note': 'linear regression fit'},
    {'Stat': 'Quadratic R2',        'Value': round(r2_quad, 4),    'Note': 'EKC quadratic fit'},
    {'Stat': 'Linear slope',        'Value': round(slope_lin, 4),  'Note': '+x = richer means more forest'},
    {'Stat': 'Quadratic coeff x2',  'Value': round(coeffs[0], 4),  'Note': '+x2 = U-shape supports EKC'},
]

df_stats = pd.DataFrame(stats_rows)
df_stats.to_csv(OUTPUT + 'stats_summary.csv', index=False, encoding='utf-8-sig')

print('Uloženo: stats_summary.csv')
print(df_stats.to_string(index=False))

Uloženo: stats_summary.csv
              Stat    Value                          Note
       N countries 199.0000       countries in Q1 dataset
        Spearman r   0.4116      log_GDP vs Forest_change
  Spearman p-value   0.0000          < 0.05 = significant
         Pearson r   0.3623      log_GDP vs Forest_change
         Linear R2   0.1312         linear regression fit
      Quadratic R2   0.1453             EKC quadratic fit
      Linear slope   1.7027 +x = richer means more forest
Quadratic coeff x2  -0.3607    +x2 = U-shape supports EKC


## 5. Příprava dat pro Q2 — Regionální panel dataset

**Výzkumná otázka Q2:**  
*Záleží na regionu více než na HDP? Jsou rozdíly mezi Afrikou, Asií, Latinskou Amerikou a Evropou větší než rozdíly dané samotným HDP?*

**Struktura výstupního souboru:**  
Jeden řádek = jedna země v jednom roce (panel data).  
Obsahuje region a income group — Power BI pak může porovnávat regiony přes čas i napříč příjmovými skupinami.

In [11]:
# Přidáme Region a Income_group do panel dat
# Panel data mají sloupec Code (ISO kód) — mergujeme na tom
q2 = df_panel.merge(df_income, on='Code', how='left')

# Odstraníme řádky bez regionu (agregátní řádky WB jako 'Africa Eastern and Southern')
q2 = q2.dropna(subset=['Region', 'Forest_share'])

# Přidáme numerické pořadí příjmové skupiny (stejná logika jako v Q1)
q2['Income_group_order'] = q2['Income_group'].map(income_order_map)

# Log transformace HDP
# replace(0, np.nan) zabrání log(0) = -nekonečno
q2['log_GDP'] = np.log(q2['GDP_per_capita'].replace(0, np.nan))

# Seřadíme: primárně podle Country, sekundárně podle roku
q2 = q2.sort_values(['Country', 'Year']).reset_index(drop=True)

q2.to_csv(OUTPUT + 'Q2_regional_panel.csv', index=False, encoding='utf-8-sig')

print(f'Uloženo: Q2_regional_panel.csv ({len(q2)} řádků)')
print('Sloupce:', q2.columns.tolist())
print('\nPokrytí regionů:')
print(q2.groupby('Region')['Country'].nunique().sort_values(ascending=False))
q2.head(3)

Uloženo: Q2_regional_panel.csv (7643 řádků)
Sloupce: ['Country', 'Code', 'Year', 'Forest_share', 'GDP_per_capita', 'Region', 'Income_group', 'Income_group_order', 'log_GDP']

Pokrytí regionů:
Region
Europe & Central Asia         55
Sub-Saharan Africa            48
Latin America & Caribbean     42
East Asia & Pacific           36
Middle East & North Africa    19
South Asia                     8
North America                  3
Name: Country, dtype: int64


,Country,Code,Year,Forest_share,GDP_per_capita,Region,Income_group,Income_group_order,log_GDP
0,Afghanistan,AFG,1990,1.854315,NaN,South Asia,Low income,1,NaN
1,Afghanistan,AFG,1991,1.854315,NaN,South Asia,Low income,1,NaN
2,Afghanistan,AFG,1992,1.854315,NaN,South Asia,Low income,1,NaN


### Regionální statistiky (přehled)

Power BI zvládne tyto výpočty pomocí DAX agregací, ale předpočítaná verze poslouží jako rychlý přehled.

In [12]:
# Průměrná změna lesa dle regionu (z Q1 cross-sectional dat)
# Každá země váží stejně — nezdůrazňujeme velké vs. malé země
regional_stats = (
    q1.groupby('Region')
    .agg(
        N_countries          = ('Country',      'count'),
        Forest_change_mean   = ('Forest_change', 'mean'),
        Forest_change_median = ('Forest_change', 'median'),
        GDP_avg_mean         = ('GDP_avg',        'mean'),
        # lambda funkce: pro každou skupinu spočítá podíl zemí s kladnou změnou
        Pct_afforesting      = ('Forest_change', lambda x: (x > 0).mean() * 100)
    )
    .round(2)
    .reset_index()
    .sort_values('Forest_change_mean')
)

print(regional_stats.to_string(index=False))

                    Region  N_countries  Forest_change_mean  Forest_change_median  GDP_avg_mean  Pct_afforesting
        Sub-Saharan Africa           46               -5.78                 -3.16       1765.67            19.57
 Latin America & Caribbean           35               -3.20                 -1.97      11253.25            22.86
       East Asia & Pacific           33               -1.10                 -0.12      11287.26            33.33
Middle East & North Africa           19                0.28                  0.00      12221.49            42.11
             North America            3                0.54                  0.71      54453.24            66.67
                South Asia            8                0.72                  0.00       1747.42            37.50
     Europe & Central Asia           55                2.75                  1.67      26733.12            80.00


## 6. Příprava dat pro Q3 — Outliers + Policy dataset

**Výzkumná otázka Q3:**  
*Kde jsou největší výjimky? Které bohaté země stále odlesňují a které chudé země naopak úspěšně zalesňují — a proč?*

**Co přidáváme:**  
- Outlier příznak (z Q1 dat — již vypočítán)
- Policy skóre: 0–4 body dle existence národní politiky, legislativy, platformy a trasovatelnosti

**Jak vzniká policy skóre:**  
Za každý sloupec s hodnotou `yes` přidáme 1 bod:
- `Policy_national` — Existuje národní politika pro udržitelné lesnictví?
- `Legislation_national` — Existuje národní legislativa?
- `Platform_national` — Existuje platforma pro zapojení stakeholders?
- `Traceability_national` — Existuje systém trasovatelnosti dřeva?

Výsledek: 0 = žádná opatření, 4 = plná institucionální podpora

In [13]:
# --- Krok 1: Výběr policy sloupců a výpočet skóre ---

policy_cols = [
    'Code', 'Policy_national', 'Legislation_national',
    'Platform_national', 'Traceability_national'
]
df_policy_q3 = df_policy[policy_cols].copy()

# Převedeme yes/no na čísla: yes -> 1, cokoliv jiného -> 0
# Iterujeme přes 4 policy sloupce a aplikujeme transformaci
for col in policy_cols[1:]:
    # .str.lower()  — normalizujeme velikost písmen (Yes, YES, yes -> yes)
    # .str.strip()  — odstraníme mezery na začátku/konci
    # == 'yes'      — porovnáme: vrátí True/False
    # .astype(int)  — True -> 1, False -> 0
    df_policy_q3[col] = (
        df_policy_q3[col].astype(str).str.lower().str.strip() == 'yes'
    ).astype(int)

# Policy skóre = součet 4 binárních sloupců (0 až 4)
df_policy_q3['Policy_score'] = (
    df_policy_q3['Policy_national'] +
    df_policy_q3['Legislation_national'] +
    df_policy_q3['Platform_national'] +
    df_policy_q3['Traceability_national']
)

# Textový label pro Power BI filtry
score_labels = {0: '0 - Zadna opatreni', 1: '1 - Minimalni',
                2: '2 - Zakladni',       3: '3 - Rozvinuta', 4: '4 - Plna'}
df_policy_q3['Policy_label'] = df_policy_q3['Policy_score'].map(score_labels)

print('Rozložení policy skóre:')
print(df_policy_q3['Policy_label'].value_counts().sort_index())

Rozložení policy skóre:
Policy_label
0 - Zadna opatreni    32
1 - Minimalni         11
2 - Zakladni          36
3 - Rozvinuta         67
4 - Plna              90
Name: count, dtype: int64


In [14]:
# --- Krok 2: Merge Q1 dat s policy skóre ---
q3 = q1.merge(df_policy_q3, on='Code', how='left')

# Příznak Is_outlier (True/False) pro snadné filtrování v Power BI
q3['Is_outlier'] = q3['Outlier_type'] != 'Expected pattern'

# Seřadíme: nejprve outliers, pak dle Forest_change
q3 = q3.sort_values(
    ['Is_outlier', 'Outlier_type', 'Forest_change'],
    ascending=[False, True, True]
).reset_index(drop=True)

q3.to_csv(OUTPUT + 'Q3_outliers_policy.csv', index=False, encoding='utf-8-sig')

print(f'Uloženo: Q3_outliers_policy.csv ({len(q3)} řádků)')
print(f'Počet outlierů: {q3["Is_outlier"].sum()}')
print('\nOutlieři:')
print(
    q3[q3['Is_outlier']]
    [['Country', 'Income_group', 'Forest_change', 'GDP_avg', 'Policy_score', 'Outlier_type']]
    .to_string(index=False)
)

Uloženo: Q3_outliers_policy.csv (199 řádků)
Počet outlierů: 7

Outlieři:
    Country        Income_group  Forest_change      GDP_avg  Policy_score       Outlier_type
      India Lower middle income           2.96  1114.969497             3 Poor & Reforesting
 Uzbekistan Lower middle income           3.05  1385.440474             3 Poor & Reforesting
      Nepal Lower middle income           3.61   591.623696             2 Poor & Reforesting
      Ghana Lower middle income           5.49  1151.398285             4 Poor & Reforesting
     Rwanda          Low income           8.01   504.067448             3 Poor & Reforesting
     Brunei         High income          -6.26 29356.312432             2 Rich & Deforesting
South Korea         High income          -2.78 20602.186752             4 Rich & Deforesting


## 7. Přehled exportovaných souborů

Rychlá kontrola, že všechny soubory byly uloženy a mají správný počet řádků.

In [15]:
output_files = [
    'Q1_ekc_crosssection.csv',
    'Q2_regional_panel.csv',
    'Q3_outliers_policy.csv',
    'stats_summary.csv',
]

print(f'{"Soubor":<35} {"Radku":>8} {"Sloupcu":>9} {"Velikost":>10}')
print('-' * 65)

for fname in output_files:
    path = OUTPUT + fname
    if os.path.exists(path):
        df_check = pd.read_csv(path, encoding='utf-8-sig')
        size_kb  = os.path.getsize(path) / 1024
        print(f'{fname:<35} {len(df_check):>8} {len(df_check.columns):>9} {size_kb:>9.1f} KB')
    else:
        print(f'{fname:<35} SOUBOR NEEXISTUJE')

Soubor                                 Radku   Sloupcu   Velikost
-----------------------------------------------------------------
Q1_ekc_crosssection.csv                  199        11      26.1 KB
Q2_regional_panel.csv                   7643         9     757.0 KB
Q3_outliers_policy.csv                   199        18      31.7 KB
stats_summary.csv                          8         3       0.4 KB


---
# Power BI — Návod k vizualizaci

---

## Krok 1 — Načtení dat

1. Otevři **Power BI Desktop**
2. Klikni na **Home -> Get Data -> Text/CSV**
3. Načti postupně všechny 4 soubory z `analysis_dan/`
4. V dialogu **Transform Data** nastav datové typy:
   - `Year` -> **Whole Number**
   - `Forest_change`, `GDP_avg`, `log_GDP` -> **Decimal Number**
   - `Income_group_order`, `Policy_score` -> **Whole Number**
5. Klikni **Close & Apply**

## Krok 2 — Nastavení relací (datový model)

Power BI propojí tabulky přes sdílené sloupce — jako FOREIGN KEY v SQL.

1. Přejdi na záložku **Model** (ikona tří čtverců vlevo)
2. Nastav tyto relace (přetáhni sloupec `Code` z jedné tabulky na druhou):

```
Q1_ekc_crosssection.Code  --(1)--(*)-- Q2_regional_panel.Code
Q1_ekc_crosssection.Code  --(1)--(*)-- Q3_outliers_policy.Code
```

- **Kardinalita:** One-to-Many (Q1 je dimenze, Q2 a Q3 jsou fact tabulky)
- **Cross-filter direction:** Single (výchozí)

## Krok 3 — Nastavení řazení Income Group

1. V záložce **Data** vyber tabulku `Q1_ekc_crosssection`
2. Klikni na sloupec **Income_group**
3. V ribbonu vyber **Column Tools -> Sort by Column -> Income_group_order**
4. Opakuj pro tabulku `Q3_outliers_policy`

## Krok 4 — DAX míry

**DAX** (Data Analysis Expressions) je jazyk pro výpočty v Power BI.  
Novou míru vytvoříš: **Home -> New Measure** (nebo pravý klik na tabulku -> New Measure)

### Základní míry pro Q1

```dax
-- Průměrná změna lesnatosti
Avg Forest Change = AVERAGE(Q1_ekc_crosssection[Forest_change])

-- Počet zemí, které zalesňují
Countries Afforesting =
    COUNTROWS(
        FILTER(Q1_ekc_crosssection, Q1_ekc_crosssection[Forest_change] > 0)
    )

-- Podíl zalesňujících zemí v procentech
-- DIVIDE(čitatel, jmenovatel, výchozí hodnota při dělení nulou)
% Afforesting =
    DIVIDE([Countries Afforesting], COUNTROWS(Q1_ekc_crosssection), 0) * 100

-- Pearsonova korelace (nativní DAX funkce)
Correlation GDP-Forest =
    PEARSON(Q1_ekc_crosssection[log_GDP], Q1_ekc_crosssection[Forest_change])
```

### Míry pro Q2 — regionální srovnání

```dax
-- Rozdíl oproti globálnímu průměru
-- ALL() odstraní aktuální filtr a spočítá průměr přes VŠECHNY řádky
Forest Change vs Global Avg =
    AVERAGE(Q1_ekc_crosssection[Forest_change]) -
    CALCULATE(
        AVERAGE(Q1_ekc_crosssection[Forest_change]),
        ALL(Q1_ekc_crosssection[Region])
    )
```

### Míry pro Q3 — outliers + policy

```dax
-- Počet outlierů v aktuálním výběru
N Outliers =
    COUNTROWS(FILTER(Q3_outliers_policy, Q3_outliers_policy[Is_outlier] = TRUE()))

-- Průměrné policy skóre pouze pro outliers
Policy Score (Outliers) =
    CALCULATE(
        AVERAGE(Q3_outliers_policy[Policy_score]),
        Q3_outliers_policy[Is_outlier] = TRUE()
    )
```

## Krok 5 — Doporučené vizuály pro každou výzkumnou otázku

---

### Q1 — Platí EKC teorie globálně?

| Vizuál | Data | Konfigurace |
|---|---|---|
| **Scatter Chart** | X: `log_GDP`, Y: `Forest_change`, Legend: `Income_group` | Hlavní EKC vizualizace. Analytics -> Polynomial trend line (degree 2) |
| **Box Plot** | Axis: `Income_group`, Values: `Forest_change` | Řadit dle `Income_group_order`. Zobrazí rozložení a mediány |
| **Card × 4** | `Avg Forest Change`, `% Afforesting`, `Correlation GDP-Forest`, `N countries` | KPI přehled v hlavičce stránky |
| **Slicer** | `Income_group` | Filtrování scatter na konkrétní skupinu |

> **Tip:** R² hodnoty ze souboru `stats_summary.csv` zobraz jako Card visual — přetáhni `Value` a nastav filtr `Stat = Quadratic R2`.

---

### Q2 — Záleží na regionu více než na HDP?

| Vizuál | Data | Konfigurace |
|---|---|---|
| **Bar Chart** | Axis: `Region`, Values: `Avg Forest Change` | Seřadit sestupně. Podmíněné formátování: červená < 0, zelená > 0 |
| **Line Chart (Small Multiples)** | Axis: `Year`, Values: `Forest_share`, Small Multiples: `Region` | Trendy v čase pro každý region zvlášť. Zdroj: Q2_regional_panel |
| **Matrix** | Rows: `Region`, Columns: `Income_group`, Values: `Avg Forest Change` | Kříž region × příjmová skupina |
| **Filled Map** | Location: `Code`, Color saturation: `Forest_change` | Gradient: červená -> bílá -> zelená |

> **Tip k mapě:** Power BI pro Filled Map potřebuje ISO kód (sloupec `Code`) nebo název země v angličtině. Sloupec `Code` je spolehlivější.

---

### Q3 — Kde jsou největší výjimky?

| Vizuál | Data | Konfigurace |
|---|---|---|
| **Scatter Chart** | X: `GDP_avg`, Y: `Forest_change`, Color: `Outlier_type`, Size: `Policy_score` | Outliers zvýrazněny barvou. Velikost bodu = síla policy |
| **Table** | `Country`, `Income_group`, `Forest_change`, `Policy_score`, `Policy_label`, `Outlier_type` | Filtrovatelná tabulka. Podmíněné formátování na `Forest_change` |
| **Bar Chart** | Axis: `Policy_label`, Values: `Avg Forest Change` | Korelace policy skóre a lesnatosti |
| **Slicer** | `Outlier_type`, `Region` | Filtrování na typ výjimky nebo region |

---

## Krok 6 — Doporučená struktura reportu

```
Strana 1: Uvod + prehled
  - Mapa sveta (Forest_change)
  - 4x Card (klicove statistiky ze stats_summary)
  - Textovy box s hypotezami

Strana 2: Q1 - EKC a HDP
  - Scatter (log_GDP x Forest_change, barva = Income_group)
  - Box plot (Income_group x Forest_change)
  - Slicer: Income group

Strana 3: Q2 - Regionalni analyza
  - Bar chart (Region x Avg Forest Change)
  - Line chart small multiples (vyvoj dle regionu)
  - Matrix (Region x Income_group)

Strana 4: Q3 - Outliers a politiky
  - Scatter (GDP_avg x Forest_change, barva = Outlier_type)
  - Bar chart (Policy_label x Avg Forest Change)
  - Tabulka outlieru
```

> **Interaktivita:** Všechny vizuály na stejné straně jsou automaticky propojené — kliknutí na region v bar chartu automaticky vyfiltruje scatter i tabulku. Funguje díky relacím nastaveným v Kroku 2.